# Tutorial on quantEM Configuration

This tutorial demonstrates how the quantEM `config` module works and how it is used to set the default computation device.

Requirements:
- An available GPU, or Apple silicon MPS (MPS is largely untested but should work)

Arthur McCray
Sep 1, 2026

In [1]:
import torch
from quantem.core import config

The most common use for `config` is setting and getting the default device. Laptops will have at maximum one gpu, or some apple devices have `mps`. In these cases it is rather simple to keep track of which device is being used, but for servers and workstations with multiple gpus it is important to specify which gpu to use.  

In [2]:
print(f"Torch cuda is available: {torch.cuda.is_available()}")
print(f"Number of GPUs available: {torch.cuda.device_count()}")
print(f"GPU names: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}")
print(f"Torch mps is available: {torch.backends.mps.is_available()}")

Torch cuda is available: True
Number of GPUs available: 4
GPU names: ['NVIDIA RTX PRO 6000 Blackwell Server Edition', 'NVIDIA RTX PRO 6000 Blackwell Server Edition', 'NVIDIA RTX PRO 6000 Blackwell Server Edition', 'NVIDIA RTX PRO 6000 Blackwell Server Edition']
Torch mps is available: False


We will proceed with selecting the first gpu (index 0)

In [3]:
print("default device: ", config.get("device"))  
config.set({"device":0})  # set device 
print("set device: ", config.get("device"))  # current device, torch string format
config.refresh()  # reset to defaults
print("device after refresh: ", config.get("device"))

default device:  cpu
set device:  cuda:0
device after refresh:  cpu


There are helpers for accessing the default compute device, as that's the most frequently used part of the config. 

You can set the default device by passing an integer (corresponding to the GPU index), a `torch.device` object, or a `torch`-style string, e.g. `"cuda:0"` to specify the first GPU.  

In [4]:
### ways to get the current device
print(f"Initial device: {config.get('device')} | {config.get_device()}")

### ways to set the device
### any of the following will work
config.set_device(0) # integer index of the gpu
config.set_device("cuda:0") # torch-style string
config.set_device(torch.device(0)) # torch device object
# config.set_device("mps") # if using an apple device with mps

print(f"device after setting: {config.get_device()}")

### reset to defaults
config.refresh()
print(f"device after refresh: {config.get_device()}")


Initial device: cpu | cpu
device after setting: cuda:0
device after refresh: cpu


The default device determines where a `torch` tensor lands if you move it with `.to("cuda")`. It is generally preferable to be explicit and send tensors to the named device with `.to(config.get_device())`.

Two other notes:
- Setting the device raises an error if there is no GPU at the index specified.
- Setting the device also selects the GPU used by `cupy`, if `cupy` is in your environment. 
    - This is a legacy effect from when `quantem` used `cupy`.

In [5]:
n_gpus = torch.cuda.device_count()
last = n_gpus - 1

print("starting quantem device: ", config.get_device())
t = torch.arange(5)
print("tensor created on: ", t.device)
t = t.to("cuda")
print("tensor moved to `cuda` lands on: ", t.device)

config.set({"device": last})
print(f"quantem device set to: {config.get_device()} (tensors are still created on cpu unless otherwise specified)")
t = t.to("cuda")
print("tensor moved to `cuda` lands on: ", t.device)
config.refresh()

starting quantem device:  cpu
tensor created on:  cpu
tensor moved to `cuda` lands on:  cuda:0
quantem device set to: cuda:3 (tensors are still created on cpu unless otherwise specified)
tensor moved to `cuda` lands on:  cuda:3


In [6]:
n_gpus = torch.cuda.device_count()
try:
    config.set_device(n_gpus)  # one past the last valid index
except RuntimeError as e:
    print(f"Only {n_gpus} GPU(s) available, so index {n_gpus} fails with:\n'''\n{e}\n'''")

Only 4 GPU(s) available, so index 4 fails with:
'''
CUDA device index 4 is out of range for 4 available devices.
'''


You can change the defaults (for this session/kernel) with `config.update_defaults`, this will not be overwritten by `refresh`

In [7]:
print(f"starting defaults: device={config.get('device')} | dtype_real={config.get('dtype_real')}")  
config.update_defaults({"device": 0})  # setting gpu with the index also works
config.set({"dtype_real": "float64"})  
print(f"updating the default device to {config.get('device')} and changing dtype_real (without updating the default) to {config.get('dtype_real')}")  
print(f"before refresh: device={config.get('device')} | dtype_real={config.get('dtype_real')}")
config.refresh()  # reset to defaults
print(f"after refresh:  device={config.get('device')} | dtype_real={config.get('dtype_real')}")
config.update_defaults({"device": "cpu"})  # setting back to original defaults


starting defaults: device=cpu | dtype_real=float32
updating the default device to cuda:0 and changing dtype_real (without updating the default) to float64
before refresh: device=cuda:0 | dtype_real=float64
after refresh:  device=cuda:0 | dtype_real=float32


#### Persisting a config change

`update_defaults` does not survive a kernel restart. It only changes what `config.refresh()` resets to.

The defaults are set on import: quantEM's own `quantem/core/quantem.yaml` is read first, then every `*.yaml` file in the user's config directory is merged on top. That directory is `~/.config/quantem/` (in line with abTEM's `~/.config/abtem/`), or wherever the `QUANTEM_CONFIG` environment variable points.

`config.write()` writes the current config to `~/.config/quantem/config.yaml`, so it is picked up on the next import. For this demo we write to a temporary directory instead, so nothing on this machine changes.

In [8]:
import tempfile
from pathlib import Path

tmp = Path(tempfile.mkdtemp())

config.set({"device": "cuda:0"})
# typically you would just run config.write() which writes to the default location
config.write(tmp / "config.yaml") 
config.refresh()
print("device after refresh: ", config.get("device"))

print("\ncontents of the written file:")
print((tmp / "config.yaml").read_text())

writing config to:  /tmp/tmpfyrgowxh/config.yaml
device after refresh:  cpu

contents of the written file:
cupy:
  fft-cache-size: 0 MB
device: cuda:0
dtype_complex: complex64
dtype_real: float32
has_cupy: false
has_torch: true
mkl:
  threads: 2
precision: float32
verbose: 1
viz:
  cmap: gray
  colors:
    paired:
    - '#3A7D44'
    - '#A2C899'
    - '#E83F85'
    - '#FFA5C5'
    - '#775AEB'
    - '#C2B4F4'
    - '#ED8607'
    - '#F9C689'
    - '#74D4B5'
    - '#C2F0DE'
    - '#808080'
    - '#C0C0C0'
    - '#C51D20'
    - '#F28E8E'
    - '#7C6A0A'
    - '#C5B86A'
    - '#00B4D8'
    - '#80D9EB'
    - '#774936'
    - '#B38B7D'
    set:
    - '#3A7D44'
    - '#E83F85'
    - '#775AEB'
    - '#ED8607'
    - '#74D4B5'
    - '#808080'
    - '#C51D20'
    - '#7C6A0A'
    - '#00B4D8'
    - '#774936'
  default_colors: ''
  interpolation: nearest
  phase_cmap: magma
  real_space_units: A
  reciprocal_space_units: A^-1
warnings:
  suppress-all-: false



`config.collect(directory)` is what import does with the user's config directory. Pointing it at the temporary directory shows what a fresh kernel would see:

In [9]:
print("device a fresh kernel would start with: ", config.collect(tmp)["device"])

device a fresh kernel would start with:  cuda:0


## The config dictionary

Everything, default or user-specified, lives in the `config.config` dictionary. `config.get` and `config.set` read and write it, and `refresh` reverts it to the defaults.

`has_cupy` reports whether `cupy` is importable, so code that optionally uses it can check before importing. `torch` is a required dependency, so `has_torch` is always true.

In [10]:
print("has cupy: ", config.get("has_cupy"))

has cupy:  False


The whole dictionary:

In [11]:
config.config

{'has_torch': True,
 'has_cupy': False,
 'device': 'cpu',
 'precision': 'float32',
 'dtype_real': 'float32',
 'dtype_complex': 'complex64',
 'verbose': 1,
 'cupy': {'fft-cache-size': '0 MB'},
 'mkl': {'threads': 2},
 'warnings': {'suppress-all-': False},
 'viz': {'interpolation': 'nearest',
  'real_space_units': 'A',
  'reciprocal_space_units': 'A^-1',
  'cmap': 'gray',
  'phase_cmap': 'magma',
  'default_colors': '',
  'colors': {'set': ['#3A7D44',
    '#E83F85',
    '#775AEB',
    '#ED8607',
    '#74D4B5',
    '#808080',
    '#C51D20',
    '#7C6A0A',
    '#00B4D8',
    '#774936'],
   'paired': ['#3A7D44',
    '#A2C899',
    '#E83F85',
    '#FFA5C5',
    '#775AEB',
    '#C2B4F4',
    '#ED8607',
    '#F9C689',
    '#74D4B5',
    '#C2F0DE',
    '#808080',
    '#C0C0C0',
    '#C51D20',
    '#F28E8E',
    '#7C6A0A',
    '#C5B86A',
    '#00B4D8',
    '#80D9EB',
    '#774936',
    '#B38B7D']}}}

The visualization defaults live under `viz`. `show_2d` and friends read these, so changing `viz.cmap` changes the default colormap everywhere.

In [12]:
config.get('viz')

{'interpolation': 'nearest',
 'real_space_units': 'A',
 'reciprocal_space_units': 'A^-1',
 'cmap': 'gray',
 'phase_cmap': 'magma',
 'default_colors': '',
 'colors': {'set': ['#3A7D44',
   '#E83F85',
   '#775AEB',
   '#ED8607',
   '#74D4B5',
   '#808080',
   '#C51D20',
   '#7C6A0A',
   '#00B4D8',
   '#774936'],
  'paired': ['#3A7D44',
   '#A2C899',
   '#E83F85',
   '#FFA5C5',
   '#775AEB',
   '#C2B4F4',
   '#ED8607',
   '#F9C689',
   '#74D4B5',
   '#C2F0DE',
   '#808080',
   '#C0C0C0',
   '#C51D20',
   '#F28E8E',
   '#7C6A0A',
   '#C5B86A',
   '#00B4D8',
   '#80D9EB',
   '#774936',
   '#B38B7D']}}

-- end -- 